In [ ]:
import yfinance as yf
import pandas as pd
import talib


In [ ]:
samsung = yf.download('005930.KS', start='2015-01-01', end='2026-03-10')

/tmp/ipykernel_55/3895182178.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  samsung = yf.download('005930.KS', start='2015-01-01', end='2026-03-10')
[*********************100%***********************]  1 of 1 completed


In [ ]:
SOX  = yf.download('^SOX',start='2020-01-01', end='2026-03-10')

/tmp/ipykernel_55/1478056237.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  SOX  = yf.download('^SOX',start='2020-01-01', end='2026-03-10')
[*********************100%***********************]  1 of 1 completed


In [ ]:
KOSPI  = yf.download('^KS11', start='2020-01-01', end='2026-03-10')

/tmp/ipykernel_55/2358091773.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  KOSPI  = yf.download('^KS11', start='2020-01-01', end='2026-03-10')
[*********************100%***********************]  1 of 1 completed


In [ ]:
df = pd.merge(pd.merge(samsung, SOX, left_index=True, right_index=True, how='inner'),KOSPI,
         left_index=True, right_index=True, how='inner')

In [ ]:
df.columns

MultiIndex([( 'Close', '005930.KS'),
            (  'High', '005930.KS'),
            (   'Low', '005930.KS'),
            (  'Open', '005930.KS'),
            ('Volume', '005930.KS'),
            ( 'Close',      '^SOX'),
            (  'High',      '^SOX'),
            (   'Low',      '^SOX'),
            (  'Open',      '^SOX'),
            ('Volume',      '^SOX'),
            ( 'Close',     '^KS11'),
            (  'High',     '^KS11'),
            (   'Low',     '^KS11'),
            (  'Open',     '^KS11'),
            ('Volume',     '^KS11')],
           names=['Price', 'Ticker'])

In [ ]:
df2 = df[[( 'Close', '005930.KS'),
            (  'High', '005930.KS'),
            (   'Low', '005930.KS'),
            (  'Open', '005930.KS'),
            ('Volume', '005930.KS'),
            ( 'Close',      '^SOX'),
            ( 'Close',     '^KS11'),
            ('Volume',     '^KS11')]].copy()

In [ ]:
df2.columns =  ['Close',	'High',	'Low',	'Open',	'Volume', 'SOX', 'KOSPI', 'KOSPI_Vol']

In [ ]:
df2['MA05'] = talib.SMA(df2['Close'], timeperiod=5)
df2['MA20'] = talib.SMA(df2['Close'], timeperiod=20)
df2['MA60'] = talib.SMA(df2['Close'], timeperiod=60)
df2['MA120'] = talib.SMA(df2['Close'], timeperiod=120)
df2['RSI_14'] = talib.RSI(df2['Close'], timeperiod=14)

In [ ]:
macd, macdsignal, macdhist = talib.MACD(df2['Close'], fastperiod=12, slowperiod=26, signalperiod=9  )
df2['macd'] = macd
df2['macdsignal'] = macdsignal
df2['macdhist'] = macdhist

In [ ]:
import numpy as np

df2['target'] = np.where(df2['Close'].shift(-1) > df2['Close'], 1, 0)

In [ ]:
df2.dropna(inplace=True)

In [ ]:
Train = df2[df2.index < '2025-12-31']
Test = df2[df2.index > '2025-12-31']

In [ ]:
df2.shape

(1344, 17)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
Train_scaler = scaler.fit_transform(Train)

In [ ]:
Test_scaler = scaler.transform(Test)

In [ ]:
Train_scaler.shape

(1302, 17)

In [ ]:
Test_scaler.shape

(42, 17)

In [ ]:
def create_seq_dataset(data, target_col_idx, window_size):
    x, y = [], []
    for   i  in range(len(data) - window_size):
        x.append(data[i : i + window_size, :-1])
        y.append(data[i + window_size, target_col_idx])
    return np.array(x), np.array(y)

In [ ]:
X, Y = create_seq_dataset(Train_scaler, -1, 20)

In [ ]:
X.shape

(1282, 20, 16)

In [ ]:
Y.shape

(1282,)

In [ ]:
import torch
x_train_tensor = torch.tensor(X, dtype=torch.float32)
y_train_tensor = torch.tensor(Y, dtype=torch.float32)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
batch_size=16
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)

In [ ]:
next(iter(train_dataset))[0].shape

torch.Size([20, 16])

In [ ]:
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class StockLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(StockLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

In [ ]:
X.shape[-1]

16

In [ ]:
input_size = X.shape[-1]
hidden_size = 64
num_layers = 1
num_classes = 2

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = StockLSTM(input_size, hidden_size, num_layers, num_classes).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
X_test, Y_test = create_seq_dataset(Test_scaler, -1, 20)
x_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(Y_test, dtype=torch.float32)


test_dataset = TensorDataset(x_test_tensor, y_test_tensor)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
val_loader = test_loader

In [ ]:
epochs = 300
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs =  inputs.to(device)
        labels = labels.squeeze().long().to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{epochs}], Loss: {running_loss / len(train_loader):.4f}')


Epoch [10/300], Loss: 0.5329
Epoch [20/300], Loss: 0.5449
Epoch [30/300], Loss: 0.5085
Epoch [40/300], Loss: 0.5090
Epoch [50/300], Loss: 0.4857
Epoch [60/300], Loss: 0.5036
Epoch [70/300], Loss: 0.4780
Epoch [80/300], Loss: 0.4727
Epoch [90/300], Loss: 0.4615
Epoch [100/300], Loss: 0.4709
Epoch [110/300], Loss: 0.4349
Epoch [120/300], Loss: 0.4288
Epoch [130/300], Loss: 0.4103


KeyboardInterrupt: 

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = StockLSTM(input_size, hidden_size, num_layers, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 150
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs =  inputs.to(device)
        labels = labels.squeeze().long().to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for val_inputs, val_labels in val_loader:
            val_inputs = val_inputs.to(device)
            val_labels = val_labels.squeeze().long().to(device)

            val_outputs = model(val_inputs)

            # loss 계산
            v_loss = criterion(val_outputs, val_labels)
            val_loss += v_loss.item()
            # 정확도
            _, predicted = torch.max(val_outputs.data, 1)
            total += val_labels.size(0)
            correct += (predicted == val_labels).sum().item()




    if (epoch + 1) % 10 == 0:
        train_loss_avg = running_loss / len(train_loader)
        val_loss_avg = val_loss / len(val_loader)
        val_accuracy = 100 * correct / total

        print(f'Epoch [{epoch + 1}/{epochs}], '
              f'Train Loss: {train_loss_avg:.4f}, '
              f'Val Loss: {val_loss_avg:.4f}, '
              f'Val Accuracy: {val_accuracy:.2f}%')


Epoch [10/150], Train Loss: 0.6872, Val Loss: 0.8182, Val Accuracy: 59.09%
Epoch [20/150], Train Loss: 0.6809, Val Loss: 0.8067, Val Accuracy: 59.09%
Epoch [30/150], Train Loss: 0.6918, Val Loss: 0.7378, Val Accuracy: 59.09%
Epoch [40/150], Train Loss: 0.6800, Val Loss: 0.8115, Val Accuracy: 59.09%
Epoch [50/150], Train Loss: 0.6820, Val Loss: 0.7785, Val Accuracy: 59.09%
Epoch [60/150], Train Loss: 0.6800, Val Loss: 0.8659, Val Accuracy: 59.09%
Epoch [70/150], Train Loss: 0.6783, Val Loss: 0.8517, Val Accuracy: 59.09%
Epoch [80/150], Train Loss: 0.6795, Val Loss: 1.0082, Val Accuracy: 59.09%
Epoch [90/150], Train Loss: 0.6795, Val Loss: 0.7472, Val Accuracy: 59.09%
Epoch [100/150], Train Loss: 0.6762, Val Loss: 1.0145, Val Accuracy: 59.09%
Epoch [110/150], Train Loss: 0.6740, Val Loss: 0.8428, Val Accuracy: 63.64%
Epoch [120/150], Train Loss: 0.6712, Val Loss: 1.0330, Val Accuracy: 59.09%
Epoch [130/150], Train Loss: 0.6752, Val Loss: 0.8463, Val Accuracy: 59.09%
Epoch [140/150], Trai

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    def forward(self, x: torch.Tensor):
        x = x + self.pe[:x.size(1)].transpose(0, 1)
        return self.dropout(x)


In [ ]:
#input_dim: 주식 데이터의 x값(특성)
#d_model :  트랜스포머의 내부 임베딩 차원
#num_heads : 멀티 헤드 어텐션의 수 - 하나만 쓰면 편향된 생각을 할 수 있음
#num_layers: 트랜스포머 인코더 레이어의 수
#output: 예측할 타켓의 차원 수

class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim: int, d_model: int, num_heads: int, num_layers: int, output_dim: int = 1, dropout: float = 0.1):
        super(TimeSeriesTransformer, self).__init__()

        self.input_projection = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.decoder = nn.Linear(d_model, output_dim)
        self.init_weights()
    def init_weights(self):
        initrange = 0.1
        self.input_projection.bias.data.zero_()
        self.input_projection.weight.data.uniform_(-initrange, initrange)
        self.decoder.bias.data.zero_()
        self.decoder.weight.data.uniform_(-initrange, initrange)

    def forward(self, src: torch.Tensor):
        src = self.input_projection(src) * math.sqrt(self.input_projection.out_features)
        src = self.pos_encoder(src)
        output = self.transformer_encoder(src)
        last_step_output = output[:, -1, :]
        prediction = self.decoder(last_step_output)
        return prediction



In [ ]:
import math
model  = TimeSeriesTransformer(input_dim=16, d_model=64, num_heads=4, num_layers=2, output_dim=1, dropout=0.1)


In [ ]:
def train_transformer_model(
    model: nn.Module,
    train_dataloader: DataLoader,
    val_dataloader: DataLoader,
    num_epochs: int = 50,
    learning_rate: float = 1e-4
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

    print(f"학습 시작: 디바이스 = {device}, 총 에포크 = {num_epochs}")

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch_idx, (X_batch, y_batch) in enumerate(train_dataloader):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            y_batch = y_batch.view(-1, 1).float()

            optimizer.zero_grad()

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * X_batch.size(0)

            predicted_labels = (outputs > 0.0).float()
            train_correct += (predicted_labels == y_batch).sum().item()
            train_total += y_batch.size(0)

        avg_train_loss = train_loss / train_total
        train_accuracy = train_correct / train_total * 100

        # 검증 단계
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for X_val, y_val in val_dataloader:
                X_val = X_val.to(device)
                y_val = y_val.to(device)
                y_val = y_val.view(-1, 1).float()


                val_outputs = model(X_val)
                loss = criterion(val_outputs, y_val)

                val_loss += loss.item() * X_val.size(0)

                val_predicted = (val_outputs > 0.0).float()
                val_correct += (val_predicted == y_val).sum().item()
                val_total += y_val.size(0)

        avg_val_loss = val_loss / val_total
        val_accuracy = val_correct / val_total * 100


        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch [{epoch + 1}/{num_epochs}] "
                  f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.2f}% || "
                  f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.2f}%")

    print("모델 학습 및 검증이 완료되었습니다.")
    return model


In [ ]:
trained_model = train_transformer_model(model, train_loader, test_loader , num_epochs=50, learning_rate=0.001)

학습 시작: 디바이스 = cuda, 총 에포크 = 50
Epoch [1/50] Train Loss: 0.7076 | Train Acc: 51.72% || Val Loss: 0.6773 | Val Acc: 59.09%
Epoch [10/50] Train Loss: 0.6949 | Train Acc: 51.40% || Val Loss: 0.7076 | Val Acc: 40.91%
Epoch [20/50] Train Loss: 0.6933 | Train Acc: 53.35% || Val Loss: 0.7108 | Val Acc: 40.91%
Epoch [30/50] Train Loss: 0.6928 | Train Acc: 53.35% || Val Loss: 0.7106 | Val Acc: 40.91%
Epoch [40/50] Train Loss: 0.6921 | Train Acc: 53.59% || Val Loss: 0.7099 | Val Acc: 40.91%
Epoch [50/50] Train Loss: 0.6920 | Train Acc: 53.35% || Val Loss: 0.7080 | Val Acc: 40.91%
모델 학습 및 검증이 완료되었습니다.
